# FashionMNIST CNN Comparison (PyTorch)

This notebook trains and compares three CNN architectures on Fashion-MNIST using PyTorch:

1. **Standard CNN** (3 conv blocks + FC)
2. **ResNet-18** (grayscale-adapted)
3. **ResNet-50** (grayscale-adapted)

Architectures are based on the descriptions in `cnn/cnn.txt`.

Key requirements implemented:

* Train/val/test split (val split tunable)
* Tunable parameters in `fas_mist_cnn()`: epochs, batch size, val split, gradient enable/disable during val/test
* Early stopping: monitor `val_accuracy`, patience=5, min_delta=0.001
* Essential validation plots: history curves, confusion matrix, sample predictions
* Timing comparison: validation/testing with gradients enabled vs disabled


In [ ]:
# If running in a fresh environment, install dependencies.
# NOTE: In many setups, torch/torchvision are already installed.

import sys
import time
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as T

import matplotlib.pyplot as plt

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
    print('CUDA capability:', torch.cuda.get_device_capability(0))


In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)


## Dataset: Fashion-MNIST

We split the original training set into train and validation using validation_split. The official test set is used as test.

In [ ]:
FASHION_CLASSES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

def get_fashionmnist_datasets(data_root: str = './data'):
    # Normalize to roughly zero mean / unit variance.
    # FashionMNIST is grayscale [0..1] after ToTensor.
    transform = T.Compose([
        T.ToTensor(),
        T.Normalize((0.5,), (0.5,))
    ])
    
    train_full = torchvision.datasets.FashionMNIST(
        root=data_root, train=True, download=True, transform=transform
    )
    test = torchvision.datasets.FashionMNIST(
        root=data_root, train=False, download=True, transform=transform
    )
    return train_full, test

train_full, test_ds = get_fashionmnist_datasets('./data')
len(train_full), len(test_ds)


In [ ]:
def split_train_val(dataset, validation_split: float = 0.1, seed: int = 42):
    assert 0.0 < validation_split < 1.0
    n = len(dataset)
    indices = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)
    val_size = int(round(n * validation_split))
    val_idx = indices[:val_size]
    train_idx = indices[val_size:]
    return Subset(dataset, train_idx.tolist()), Subset(dataset, val_idx.tolist())

train_ds, val_ds = split_train_val(train_full, validation_split=0.1, seed=42)
len(train_ds), len(val_ds)


## Models
We implement three architectures described in cnn/cnn.txt:
• Standard CNN (3 conv blocks)
• ResNet-18 (1-channel input)
• ResNet-50 (1-channel input)

In [ ]:
class StandardCNN(nn.Module):
    """Standard CNN for 1x28x28 grayscale images (FashionMNIST)."""
    
    def __init__(self, num_classes: int = 10, dropout: float = 0.25):
        super().__init__()
        
        # Block 1
        self.conv1a = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1a = nn.BatchNorm2d(32)
        self.conv1b = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn1b = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.drop1 = nn.Dropout(dropout)
        
        # Block 2
        self.conv2a = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2a = nn.BatchNorm2d(64)
        self.conv2b = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2b = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.drop2 = nn.Dropout(dropout)
        
        # Block 3 (optional in description; we include it to match the "complex" variant)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        self.drop3 = nn.Dropout(dropout)
        # FC
        # After 3 pools: 28 -> 14 -> 7 -> 3 (floor)
        # With padding=1 convs, spatial dims preserved before pooling.
        # 28/2=14, 14/2=7, 7/2=3
        self.fc1 = nn.Linear(128 * 3 * 3, 128)
        self.drop_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = F.relu(self.bn1b(self.conv1b(x)))
        x = self.pool1(x)
        x = self.drop1(x)
        
        x = F.relu(self.bn2a(self.conv2a(x)))
        x = F.relu(self.bn2b(self.conv2b(x)))
        x = self.pool2(x)
        x = self.drop2(x)
        
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool3(x)
        x = self.drop3(x)
        
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.drop_fc(x)
        x = self.fc2(x)
        return x


In [ ]:
def make_resnet(model_name: str, num_classes: int = 10) -> nn.Module:
    """Create a grayscale-adapted ResNet from torchvision."""
    if model_name == 'resnet18':
        model = torchvision.models.resnet18(weights=None)
    elif model_name == 'resnet50':
        model = torchvision.models.resnet50(weights=None)
    else:
        raise ValueError(f'Unknown model_name: {model_name}')
        
    # Modify first conv to accept 1 channel instead of 3.
    # Original: Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
    old_conv = model.conv1
    model.conv1 = nn.Conv2d(
        in_channels=1,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=old_conv.bias is not None
    )
    
    # Replace classifier
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

res18 = make_resnet('resnet18')
res50 = make_resnet('resnet50')
sum(p.numel() for p in res18.parameters()), sum(p.numel() for p in res50.parameters())


## EarlyStopping

EarlyStopping (monitor=val_accuracy)

Stops training when validation accuracy does not improve by at least min_delta for patience epochs.

In [ ]:
@dataclass
class EarlyStopping:
    monitor: str = 'val_accuracy'
    patience: int = 5
    min_delta: float = 0.001
    
    best: float = -float('inf')
    bad_epochs: int = 0
    best_state_dict: Optional[Dict[str, torch.Tensor]] = None
    
    def step(self, metrics: Dict[str, float], model: nn.Module) -> bool:
        """Returns True if training should stop."""
        current = float(metrics[self.monitor])
        if current > self.best + self.min_delta:
            self.best = current
            self.bad_epochs = 0
            # Keep best weights in memory
            self.best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            return False
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore_best(self, model: nn.Module) -> None:
        if self.best_state_dict is not None:
            model.load_state_dict(self.best_state_dict)


Training / Validation Utilities
We keep a tunable switch disable_grad_in_eval to compare:

* Normal validation/testing: torch.no_grad() (gradients disabled)
* With gradients enabled: no no_grad() context (slower, higher memory)

This is useful to see the effect on val_loss/val_accuracy curves and timing.



In [ ]:
def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == targets).float().mean().item()

def run_one_epoch_train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0
    
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), y)
        n_batches += 1
        
    return {
        'train_loss': total_loss / max(1, n_batches),
        'train_accuracy': total_acc / max(1, n_batches)
    }
    
def run_one_epoch_eval(model, loader, criterion, device, disable_grad_in_eval: bool = True):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0
    
    # If disable_grad_in_eval=True, we use torch.no_grad() (recommended).
    # If False, gradients are enabled (for experimentation / timing comparison).
    ctx = torch.no_grad() if disable_grad_in_eval else torch.enable_grad()
    
    start = time.perf_counter()
    with ctx:
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            
            logits = model(x)
            loss = criterion(logits, y)
            
            total_loss += loss.item()
            total_acc += accuracy_from_logits(logits.detach(), y)
            n_batches += 1
            
    elapsed = time.perf_counter() - start
    
    return {
        'val_loss': total_loss / max(1, n_batches),
        'val_accuracy': total_acc / max(1, n_batches),
        'eval_seconds': elapsed
    }


# Plots

1. Training history curves (loss/accuracy)
2. Confusion matrix
3. Sample predictions


In [ ]:
def plot_history(history: Dict[str, List[float]], title: str = ''):
    epochs = np.arange(1, len(history['train_loss']) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title)
    
    axes[0].plot(epochs, history['train_loss'], label='train_loss')
    axes[0].plot(epochs, history['val_loss'], label='val_loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True)
    axes[0].legend()
    
    axes[1].plot(epochs, history['train_accuracy'], label='train_accuracy')
    axes[1].plot(epochs, history['val_accuracy'], label='val_accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].grid(True)
    axes[1].legend()
    
    plt.show()

def confusion_matrix(model, loader, device, disable_grad_in_eval: bool = True) -> np.ndarray:
    model.eval()
    cm = np.zeros((10, 10), dtype=np.int64)
    ctx = torch.no_grad() if disable_grad_in_eval else torch.enable_grad()
    with ctx:
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            preds = torch.argmax(logits, dim=1)
            for t, p in zip(y.view(-1).cpu().numpy(), preds.view(-1).cpu().numpy()):
                cm[t, p] += 1
                
    return cm

def plot_confusion_matrix(cm: np.ndarray, class_names: List[str], title: str = 'Confusion Matrix'):
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel='True label',
        xlabel='Predicted label',
        title=title
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
    
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'),
                    ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=8)
            
    fig.tight_layout()
    plt.show()

def plot_sample_predictions(model, dataset, device, disable_grad_in_eval: bool = True, n: int = 12, seed: int = 0, title: str = ''):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(dataset), size=n, replace=False)
    
    model.eval()
    ctx = torch.no_grad() if disable_grad_in_eval else torch.enable_grad()
    
    cols = 6
    rows = int(math.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.2))
    axes = np.array(axes).reshape(-1)
    fig.suptitle(title)
    
    with ctx:
        for ax_i, ds_i in enumerate(idxs):
            x, y = dataset[ds_i]
            x_b = x.unsqueeze(0).to(device)
            logits = model(x_b)
            pred = int(torch.argmax(logits, dim=1).item())
            
            img = x.squeeze(0).cpu().numpy()
            axes[ax_i].imshow(img, cmap='gray')
            axes[ax_i].axis('off')
            axes[ax_i].set_title(f'T:{FASHION_CLASSES[int(y)]}\nP:{FASHION_CLASSES[pred]}', fontsize=9)
            
    for j in range(len(idxs), len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()


# Main API: fas_mist_cnn()

This function:
* Builds the selected model
* Trains with early stopping
* Evaluates on validation and test
* Produces required plots
* Measures timing for eval with gradients enabled/disabled

Tunable parameters:
* `epochs` (default 50)
* `batch_size` (default 64)
* `validation_split` (default 0.1)
* `disable_grad_in_eval` (default True)


In [ ]:
def build_model(arch: str, num_classes: int = 10) -> nn.Module:
    arch = arch.lower()
    if arch in ['standard', 'cnn', 'standard_cnn']:
        return StandardCNN(num_classes=num_classes)
    if arch in ['resnet18', 'resnet-18']:
        return make_resnet('resnet18', num_classes=num_classes)
    if arch in ['resnet50', 'resnet-50']:
        return make_resnet('resnet50', num_classes=num_classes)
    raise ValueError(f'Unknown arch: {arch}')

def fas_mist_cnn(
    arch: str = 'standard',
    epochs: int = 50,
    batch_size: int = 64,
    validation_split: float = 0.1,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    disable_grad_in_eval: bool = True,
    early_stopping: Optional[EarlyStopping] = None,
    num_workers: int = 2,
    pin_memory: bool = True,
    seed: int = 42
) -> dict:
    """Train and validate a FashionMNIST model.
    
    Args:
        arch: 'standard' | 'resnet18' | 'resnet50'
        epochs: default 50
        batch_size: default 64
        validation_split: default 0.1
        lr: learning rate (tunable)
        weight_decay: L2 regularization (tunable)
        disable_grad_in_eval: True -> torch.no_grad() in val/test; False -> gradients enabled
        early_stopping: EarlyStopping instance (defaults to monitor val_accuracy)
    """
    set_seed(seed)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    train_full, test_ds = get_fashionmnist_datasets('./data')
    train_ds, val_ds = split_train_val(train_full, validation_split=validation_split, seed=seed)
    
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory
    )
    
    model = build_model(arch).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    if early_stopping is None:
        early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, min_delta=0.001)
    history = {
        'train_loss': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_accuracy': [],
        'eval_seconds': []
    }
    
    print('Device:', device)
    print('Arch:', arch)
    print(f'Epochs: {epochs}, batch_size: {batch_size}, val_split: {validation_split}')
    print(f'lr: {lr}, weight_decay: {weight_decay}')
    print('disable_grad_in_eval:', disable_grad_in_eval)
    
    for epoch in range(1, epochs + 1):
        train_metrics = run_one_epoch_train(model, train_loader, optimizer, criterion, device)
        val_metrics = run_one_epoch_eval(model, val_loader, criterion, device, disable_grad_in_eval=disable_grad_in_eval)
        
        for k in ['train_loss', 'train_accuracy']:
            history[k].append(train_metrics[k])
        for k in ['val_loss', 'val_accuracy', 'eval_seconds']:
            history[k].append(val_metrics[k])
            
        print(
            f"Epoch {epoch:03d}/{epochs} | "
            f"train_loss={train_metrics['train_loss']:.4f} train_acc={train_metrics['train_accuracy']:.4f} | "
            f"val_loss={val_metrics['val_loss']:.4f} val_acc={val_metrics['val_accuracy']:.4f} | "
            f"val_time={val_metrics['eval_seconds']:.2f}s"
        )
        
        should_stop = early_stopping.step({'val_accuracy': val_metrics['val_accuracy']}, model)
        if should_stop:
            print(
                f"EarlyStopping triggered at epoch {epoch}. "
                f"Best val_accuracy={early_stopping.best:.4f}"
            )
            break
            
    early_stopping.restore_best(model)
    
    # # Final evaluation on test
    test_metrics = run_one_epoch_eval(model, test_loader, criterion, device, disable_grad_in_eval=disable_grad_in_eval)
    print(f"Test: loss={test_metrics['val_loss']:.4f} acc={test_metrics['val_accuracy']:.4f} time={test_metrics['eval_seconds']:.2f}s")
    
    # Plots
    plot_history(history, title=f"{arch} | disable_grad_in_eval={disable_grad_in_eval}")
    
    cm = confusion_matrix(model, test_loader, device, disable_grad_in_eval=disable_grad_in_eval)
    plot_confusion_matrix(cm, FASHION_CLASSES, title=f"Test Confusion Matrix | {arch}")
    
    plot_sample_predictions(
        model, test_ds, device,
        disable_grad_in_eval=disable_grad_in_eval,
        n=12,
        seed=0,
        title=f"Sample Predictions (Test) | {arch}"
    )
    
    return {
        'model': model,
        'history': history,
        'test_metrics': test_metrics,
        'config': {
            'arch': arch,
            'epochs': epochs,
            'batch_size': batch_size,
            'validation_split': validation_split,
            'lr': lr,
            'weight_decay': weight_decay,
            'disable_grad_in_eval': disable_grad_in_eval,
        }
    }


# Compare Models

Runs all three architectures and compares:
* Best validation accuracy
* Test accuracy
* Validation time per epoch

Also compares eval with gradients enabled vs disabled.


In [ ]:
def summarize_run(run_result: Dict) -> Dict[str, float]:
    hist = run_result['history']
    best_val_acc = float(np.max(hist['val_accuracy'])) if len(hist['val_accuracy']) else float('nan')
    best_val_loss = float(np.min(hist['val_loss'])) if len(hist['val_loss']) else float('nan')
    avg_val_time = float(np.mean(hist['eval_seconds'])) if len(hist['eval_seconds']) else float('nan')
    test_acc = float(run_result['test_metrics']['val_accuracy'])
    test_loss = float(run_result['test_metrics']['val_loss'])
    test_time = float(run_result['test_metrics']['eval_seconds'])
    return {
        'best_val_acc': best_val_acc,
        'best_val_loss': best_val_loss,
        'avg_val_time_s': avg_val_time,
        'test_acc': test_acc,
        'test_loss': test_loss,
        'test_time_s': test_time,
    }

def compare_architectures(
    epochs: int = 50,
    batch_size: int = 64,
    validation_split: float = 0.1,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
):
    results = []
    for arch in ['standard', 'resnet18', 'resnet50']:
        for disable_grad_in_eval in [True, False]:
            print('\n' + '=' * 80)
            print(f"RUN: arch={arch} disable_grad_in_eval={disable_grad_in_eval}")
            run = fas_mist_cnn(
                arch=arch,
                epochs=epochs,
                batch_size=batch_size,
                validation_split=validation_split,
                lr=lr,
                weight_decay=weight_decay,
                disable_grad_in_eval=disable_grad_in_eval,
            )
            summary = summarize_run(run)
            summary.update({'arch': arch, 'disable_grad_in_eval': disable_grad_in_eval})
            results.append(summary)
            
    # Pretty print
    import pandas as pd
    df = pd.DataFrame(results)
    df = df[['arch', 'disable_grad_in_eval', 'best_val_acc', 'best_val_loss', 'avg_val_time_s', 'test_acc', 'test_loss', 'test_time_s']]
    return df

# Example usage (can take time on ResNet50):
# df = compare_architectures(epochs=10, batch_size=64, validation_split=0.1, lr=1e-3)
# df


# Notes on the validation behavior you described

You mentioned:

* Accuracy jagged (0.94-0.98)
* Loss high initially (>2.0), noisy
* Large fluctuations between train/val

Common causes:

* Learning rate too high (try `lr=3e-4` or `1e-4`)
* Model too complex (ResNet50 on FashionMNIST can overfit quickly)
* Regularization too strong or too weak (tune `weight_decay`, `dropout`)
* Batch size too small (noisy gradients)

This notebook keeps `lr`, `weight_decay`, and `disable_grad_in_eval` tunable so you can reproduce and study these effects.
